# 🧬 PokéMatch — Colab 준비/학습 노트북

이 노트북은 **"닮은 포켓몬 찾기"** 앱에 필요한 웹 자산을 만든다.

**접근:** 분류 학습이 아니라 **임베딩 + 최근접 이웃**.
사전학습 이미지 인코더로 포켓몬 대표 벡터(갤러리)를 미리 계산하고, 웹에서는 얼굴 crop을
같은 인코더로 임베딩해 코사인 유사도로 상위 5개를 찾는다.

**이 노트북이 만드는 산출물**
| 파일 | 용도 | 배치 |
|---|---|---|
| `pokemon_encoder.int8.onnx` | 브라우저 임베딩(onnxruntime-web) | 웹 `/public/models/` 또는 B2 |
| `gallery.bin` + `gallery.json` | 1069종 프로토타입 벡터(int8) + 메타 | 웹 `/public/pokemon/` |
| `pokedex.json` | 도감번호·한글명·영문명·타입 | 웹 `/public/pokemon/` |
| `pokedex_webp/*.webp` | 결과용 대표 이미지(축소) | **B2 호스팅** |

> ⚠️ **전처리 일치가 1순위.** 5번 셀이 출력하는 입력크기·mean·std를 웹 JS 전처리에 그대로 옮겨야 한다.


In [ ]:
#@title 1. 의존성 설치 (onnxscript 제거가 핵심)
# ⚠️ 절대 금지: numpy 강제변경('Numpy is not available'), facenet-pytorch(numpy 다운그레이드).
# onnxscript가 깔려 있으면 Colab torch가 import 중 그걸 참조하다 깨진다(ParamSchema 없음).
# 우리는 레거시 ONNX exporter(dynamo=False)라 onnxscript가 필요 없으므로 맨 먼저 제거한다.
!pip uninstall -y onnxscript
!pip -q install open_clip_torch onnx onnxruntime
import torch, open_clip, numpy as np
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("open_clip", getattr(open_clip, "__version__", "?"), "| numpy", np.__version__)
print("bridge OK:", torch.from_numpy(np.zeros(3)).shape)   # 에러 없이 (3,) 나오면 완전 정상


In [ ]:
#@title 2. 설정 (CONFIG)
import os

# --- 인코더 선택 ---------------------------------------------------------
# 기본: 온디바이스용 경량 CLIP. 사용 중인 open_clip 버전에 없으면 아래 5번 셀이
# 사용 가능한 태그를 출력해 주므로 그걸로 교체하면 된다.
ENCODER_NAME      = "MobileCLIP2-S0"    # 이 open_clip 버전 기준 사용 가능 태그. 대안: "MobileCLIP-S1"
ENCODER_PRETRAIN  = "dfndr2b"           # MobileCLIP-S1이면 "datacompdr" / ViT-B-32면 "laion2b_s34b_b79k"

# --- 갤러리 생성 파라미터 -------------------------------------------------
N_PER_SPECIES = 150     # 종당 임베딩에 쓸 최대 이미지 수(다양한 화풍 평균 → 스타일 불변)
BATCH         = 64
L2_NORMALIZE  = True    # 임베딩 L2 정규화(코사인 유사도용) — 반드시 웹과 동일하게

# --- 출력 위치 -----------------------------------------------------------
# Drive에 두면 세션이 끊겨도 산출물이 남는다.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = "/content/drive/MyDrive/pokematch_out"
else:
    OUT = "/content/pokematch_out"
os.makedirs(OUT, exist_ok=True)
print("출력 폴더:", OUT)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. 데이터 확보 — 둘 중 하나

- **3-A Kaggle 직접 다운로드(권장)**: 클라우드 내려받기라 빠르다. `kaggle.json` 필요.
- **3-B Drive 업로드**: 로컬 `archive/`를 구글 드라이브에 올려두고 경로만 지정.


In [ ]:
#@title 3-A. Kaggle 다운로드 (kaggle.json 업로드)
# Kaggle 계정 > Settings > API > Create New Token 으로 받은 kaggle.json 업로드
from google.colab import files
import os, json
up = files.upload()               # kaggle.json 선택
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip -q install kaggle
!kaggle datasets download -d shivanshcoding/1282-pokemon-139542-images-updated-pokedex-dataset -p /content
!mkdir -p /content/archive && unzip -q -o /content/*.zip -d /content/archive
DATA_ROOT = "/content/archive"
print("DATA_ROOT =", DATA_ROOT)


In [ ]:
#@title 3-B. (대안) Drive에 올린 archive 경로 지정
# DATA_ROOT = "/content/drive/MyDrive/archive"
# print(DATA_ROOT)


In [ ]:
#@title 4. 데이터 폴더 자동 탐색
import os, glob
def find_dir(root, *needles):
    for d,_,_ in os.walk(root):
        base = os.path.basename(d).lower()
        if all(n in base for n in needles): return d
    return None

REPR_DIR  = find_dir(DATA_ROOT, "pokedex", "image")        # 대표 이미지 1장/종
CLS_DIR   = find_dir(DATA_ROOT, "classification")          # 종 폴더(다수 이미지)
CSV_PATH  = glob.glob(os.path.join(DATA_ROOT, "**", "*pokedex*dataset*.csv"), recursive=True)
CSV_PATH  = CSV_PATH[0] if CSV_PATH else None
print("REPR_DIR:", REPR_DIR)
print("CLS_DIR :", CLS_DIR)
print("CSV_PATH:", CSV_PATH)
species = sorted([d for d in os.listdir(CLS_DIR) if os.path.isdir(os.path.join(CLS_DIR,d))])
print("종 개수:", len(species), "| 예:", species[:5])


In [ ]:
#@title 5. 인코더 로드 + 전처리 상수 출력 (★ 웹과 일치시킬 값)
import open_clip, torch

# 사용 가능한 MobileCLIP 태그 확인(내 기본값이 안 맞을 때 참고)
print("MobileCLIP 관련 사용 가능 태그:")
for m,p in open_clip.list_pretrained():
    if "mobileclip" in m.lower(): print("  ", m, p)

model, _, preprocess = open_clip.create_model_and_transforms(
    ENCODER_NAME, pretrained=ENCODER_PRETRAIN)
model = model.eval().to(DEVICE)

# open_clip preprocess에서 실제 입력 크기/정규화 값을 추출 → 웹 전처리에 그대로 사용
import torchvision.transforms as T
size = None; crop = None; mean = None; std = None
for t in preprocess.transforms:
    if isinstance(t, T.Resize):          size = t.size
    if isinstance(t, T.CenterCrop):      crop = t.size
    if isinstance(t, T.Normalize):       mean, std = list(t.mean), list(t.std)
# 정사각 입력 크기(정수) — ONNX dummy와 웹 캔버스 크기에 사용
IMG_SIZE = (crop[0] if isinstance(crop,(tuple,list)) else crop) or \
           (size[0] if isinstance(size,(tuple,list)) else size)
print("\n★ 웹 전처리 상수 ---------------------------")
print("resize:", size, "| center-crop:", crop, "| IMG_SIZE:", IMG_SIZE)
print("mean:", mean)
print("std :", std)
print("embedding dim:", getattr(model.visual, 'output_dim', 'unknown'))


In [ ]:
#@title 6. 갤러리 임베딩 생성 (종별 프로토타입)
import os, torch, numpy as np
from PIL import Image
from tqdm.auto import tqdm

@torch.no_grad()
def embed_paths(paths):
    outs = []
    for i in range(0, len(paths), BATCH):
        batch = []
        for p in paths[i:i+BATCH]:
            try: batch.append(preprocess(Image.open(p).convert("RGB")))
            except Exception: pass
        if not batch: continue
        x = torch.stack(batch).to(DEVICE)
        f = model.encode_image(x).float()
        if L2_NORMALIZE: f = torch.nn.functional.normalize(f, dim=-1)
        outs.append(f.cpu())
    return torch.cat(outs) if outs else None

protos, kept = [], []
for sp in tqdm(species):
    files = sorted(os.listdir(os.path.join(CLS_DIR, sp)))[:N_PER_SPECIES]
    paths = [os.path.join(CLS_DIR, sp, f) for f in files]
    emb = embed_paths(paths)
    if emb is None:
        print("skip(빈 폴더):", sp); continue
    proto = torch.nn.functional.normalize(emb.mean(0), dim=-1)  # 평균 후 재정규화
    protos.append(proto.numpy()); kept.append(sp)

gallery = np.stack(protos).astype(np.float32)   # [S, D]
print("gallery:", gallery.shape, "| 종:", len(kept))


In [ ]:
#@title 7. 갤러리 저장 + int8 양자화(웹용)
import json, numpy as np, os
np.save(os.path.join(OUT, "gallery.npy"), gallery)

# L2 정규화된 벡터라 값이 [-1,1] → 전역 스케일 127로 int8 양자화
SCALE = 127.0
q = np.clip(np.round(gallery * SCALE), -127, 127).astype(np.int8)
q.tofile(os.path.join(OUT, "gallery.bin"))
with open(os.path.join(OUT, "gallery.json"), "w", encoding="utf-8") as f:
    json.dump({"species": kept, "dim": int(gallery.shape[1]),
               "count": int(gallery.shape[0]), "quant": "int8", "scale": SCALE,
               "l2_normalized": bool(L2_NORMALIZE)}, f, ensure_ascii=False)
print("saved gallery.bin", q.nbytes, "bytes")


In [ ]:
#@title 7-B. ★인기편향 통계 μ_p, σ_p (z-score 재랭킹용) — 필수
# '파라스처럼 모든 얼굴과 가까운' 포켓몬이 항상 1등 되는 편향 제거의 핵심.
# 일반 얼굴 집단(LFW) 대비 각 포켓몬의 유사도 평균/표준편차를 구해 gallery.json에 저장.
import numpy as np, torch, json, os
from PIL import Image
from sklearn.datasets import fetch_lfw_people
dev = next(model.parameters()).device
lfw = fetch_lfw_people(color=True, resize=1.0, funneled=True)   # 최초 1회 ~200MB
idx = np.random.default_rng(0).choice(len(lfw.images), size=min(500, len(lfw.images)), replace=False)
F = []
with torch.no_grad():
    for i in idx:
        im = Image.fromarray((lfw.images[i]*255).astype("uint8")).convert("RGB")
        e = model.encode_image(preprocess(im).unsqueeze(0).to(dev)).float()
        F.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy()[0])
F = np.stack(F)
A = F @ gallery.T                       # [얼굴수, 종수]
mu_p = A.mean(0).astype(float); sd_p = (A.std(0) + 1e-6).astype(float)

gp = os.path.join(OUT, "gallery.json")
meta = json.load(open(gp, encoding="utf-8"))
meta["rerank"] = "zscore"; meta["mu"] = mu_p.tolist(); meta["sd"] = sd_p.tolist()
json.dump(meta, open(gp, "w", encoding="utf-8"), ensure_ascii=False)
print("μ_p, σ_p 저장 완료 (faces:", len(F), ") → gallery.json")


In [ ]:
#@title 8. 인코더 ONNX 변환 + int8 양자화 + 정합성 검증
import torch, os, numpy as np, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

class ImageEmbedder(torch.nn.Module):
    def __init__(self, m, l2): super().__init__(); self.m=m; self.l2=l2
    def forward(self, x):
        f = self.m.encode_image(x)
        if self.l2: f = torch.nn.functional.normalize(f, dim=-1)
        return f

# ⚠️ model을 .to("cpu")로 옮기지 말 것(갤러리/13번 셀이 쓰는 동일 객체까지 CPU로 감).
#    모델의 현재 device를 그대로 따라간다.
dev = next(model.parameters()).device
wrapper = ImageEmbedder(model, L2_NORMALIZE).eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(dev)
with torch.no_grad(): ref = wrapper(dummy).cpu().numpy()
print("embedding dim:", ref.shape[-1])

def cosine(a, b): return float((a*b).sum() / (np.linalg.norm(a)*np.linalg.norm(b)))
def onnx_cos(path):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    name = s.get_inputs()[0].name          # exporter마다 입력 이름이 다를 수 있음
    return cosine(ref, s.run(None, {name: dummy.cpu().numpy()})[0])

fp32 = os.path.join(OUT, "pokemon_encoder.onnx")
int8 = os.path.join(OUT, "pokemon_encoder.int8.onnx")

# 배치=1 고정(dynamic_axes 제거) — 얼굴 1장만 추론하므로 충분하고, 동적축이 유발한
# ShapeInferenceError를 피한다.
def export(use_dynamo):
    if use_dynamo:   # 신규 torch.export 기반(FastViT 재현도가 더 나을 수 있음)
        torch.onnx.export(wrapper, dummy, fp32,
            input_names=["pixel_values"], output_names=["embedding"], dynamo=True)
    else:            # 레거시 TorchScript
        torch.onnx.export(wrapper, dummy, fp32, opset_version=17,
            input_names=["pixel_values"], output_names=["embedding"],
            do_constant_folding=True, dynamo=False)

export(False); c_fp32 = onnx_cos(fp32); print("fp32(legacy) 코사인:", round(c_fp32,4))
if c_fp32 <= 0.99:
    print("→ 부정확. 신규 exporter(dynamo=True)로 재시도"); export(True)
    c_fp32 = onnx_cos(fp32); print("fp32(dynamo) 코사인:", round(c_fp32,4))

# int8 동적 양자화는 FastViT에서 정확도가 깨질 수 있음 → 검증 후 통과할 때만 채택
c_int8 = None
try:
    quantize_dynamic(fp32, int8, weight_type=QuantType.QInt8)
    c_int8 = onnx_cos(int8); print("int8 코사인:", round(c_int8,4))
except Exception as e:
    print("int8 양자화 실패:", e)

if c_int8 is not None and c_int8 > 0.99:
    WEB_MODEL = int8
else:
    WEB_MODEL = fp32
    if os.path.exists(int8): os.remove(int8)
    if c_int8 is not None: print("int8 정확도 불량 → 삭제하고 fp32 사용")
print("웹 배포 모델:", os.path.basename(WEB_MODEL), "|", os.path.getsize(WEB_MODEL), "bytes",
      "| ✅ OK" if c_fp32 > 0.99 else "| ⚠️ 여전히 부정확(로그 공유 요망)")


In [ ]:
#@title 8-B. (권장) fp16 변환 — 모델 ~절반 크기, 로딩 빨라짐
# 입출력은 float32 유지(keep_io_types)라 웹 전처리/코드는 그대로. 내부 가중치만 fp16.
# ⚠️ cell 8 직후 실행(ref, dummy 재사용). fp16이 웹(onnxruntime-web)에서 문제되면 fp32로 되돌리면 됨.
!pip -q install onnxconverter-common
import onnx, os, numpy as np, onnxruntime as ort
from onnxconverter_common import float16
p = os.path.join(OUT, "pokemon_encoder.onnx")
before = os.path.getsize(p)
onnx.save(float16.convert_float_to_float16(onnx.load(p), keep_io_types=True), p)  # 같은 파일명으로 덮어씀
after = os.path.getsize(p)
s = ort.InferenceSession(p, providers=["CPUExecutionProvider"])
got = s.run(None, {s.get_inputs()[0].name: dummy.cpu().numpy()})[0]
cos = float((ref*got).sum()/(np.linalg.norm(ref)*np.linalg.norm(got)))
print(f"fp32 {before//1024//1024}MB → fp16 {after//1024//1024}MB | torch 대비 코사인 {cos:.4f}",
      "→ OK" if cos>0.99 else "→ 확인 필요")


In [ ]:
#@title 9. 메타데이터 pokedex.json (타입=CSV, 도감번호·한글명=PokéAPI best-effort)
import csv, json, os, re, time, requests
from tqdm.auto import tqdm

def slug(s): return re.sub(r'[^a-z0-9]+','_', s.lower()).strip('_')

TYPE_KO = {"normal":"노말","fire":"불꽃","water":"물","grass":"풀","electric":"전기",
    "ice":"얼음","fighting":"격투","poison":"독","ground":"땅","flying":"비행",
    "psychic":"에스퍼","bug":"벌레","rock":"바위","ghost":"고스트","dragon":"드래곤",
    "dark":"악","steel":"강철","fairy":"페어리"}

# CSV → 타입(신뢰) 매핑 (slug 기준)
csv_types = {}
with open(CSV_PATH, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        s = slug(row["Name"])
        ts = [t for t in [row.get("Type1","").strip(), row.get("Type2","").strip()] if t]
        csv_types[s] = ts

# PokéAPI 정규화(언더스코어→하이픈, 흔한 예외)
def api_name(s):
    n = s.replace("_","-")
    fix = {"ho-oh":"ho-oh","farfetch-d":"farfetchd","mr-mime":"mr-mime",
           "nidoran-f":"nidoran-f","nidoran-m":"nidoran-m","porygon-z":"porygon-z"}
    n = re.sub(r"^mega-","", n)   # 메가는 기본종으로 조회
    return fix.get(n, n)

cache_path = os.path.join(OUT, "_pokeapi_cache.json")
cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
def poke(name):
    if name in cache: return cache[name]
    try:
        r = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{name}", timeout=10)
        cache[name] = r.json() if r.ok else None
    except Exception: cache[name] = None
    return cache[name]

pokedex = {}
for sp in tqdm(kept):
    types_en = csv_types.get(sp, [])
    entry = {"slug": sp, "nameEn": sp.replace("_"," ").title(),
             "typesEn": types_en, "typesKo": [TYPE_KO.get(t.lower(), t) for t in types_en],
             "dex": None, "nameKo": None, "isMega": sp.startswith("mega_"),
             "color": None, "shape": None}   # 속성 재랭킹용(PokéAPI 무료 제공)
    data = poke(api_name(sp))
    if data:
        entry["dex"] = data.get("id")
        entry["color"] = (data.get("color") or {}).get("name")   # red/blue/green/brown/...
        entry["shape"] = (data.get("shape") or {}).get("name")   # ball/upright/humanoid/...
        for nm in data.get("names", []):
            if nm.get("language",{}).get("name") == "ko":
                entry["nameKo"] = nm["name"]; break
    pokedex[sp] = entry

json.dump(cache, open(cache_path,"w"), ensure_ascii=False)
json.dump(pokedex, open(os.path.join(OUT,"pokedex.json"),"w",encoding="utf-8"),
          ensure_ascii=False, indent=0)
miss_dex = sum(1 for e in pokedex.values() if e["dex"] is None)
miss_ko  = sum(1 for e in pokedex.values() if e["nameKo"] is None)
print(f"총 {len(pokedex)}종 | 도감번호 미확인 {miss_dex} | 한글명 미확인 {miss_ko}")
print("예:", json.dumps(pokedex[kept[0]], ensure_ascii=False))


In [ ]:
#@title 9-B. 기본종만 남기기 (메가·패러독스·특수폼 제외) — 웹 자산 필터
# 사용자 지시: 결과는 기본종만. 갤러리/통계/메타/도감이미지를 base-only로 필터링해 저장.
import numpy as np, json, os
PARADOX = {"great_tusk","scream_tail","brute_bonnet","flutter_mane","slither_wing",
           "sandy_shocks","roaring_moon","walking_wake","gouging_fire","raging_bolt"}
def is_base(s):
    return not (s.startswith(("mega_","primal_","gmax_","iron_")) or s in PARADOX)

gj = os.path.join(OUT,"gallery.json")
meta = json.load(open(gj, encoding="utf-8"))
gallery = np.load(os.path.join(OUT,"gallery.npy")).astype("float32")
kept = meta["species"]
mu = np.array(meta["mu"]); sd = np.array(meta["sd"])
keep = [i for i,s in enumerate(kept) if is_base(s)]
print(f"기본종 {len(keep)} / 전체 {len(kept)} (제외 {len(kept)-len(keep)})")

kept = [kept[i] for i in keep]
gallery = gallery[keep]; mu = mu[keep]; sd = sd[keep]
np.save(os.path.join(OUT,"gallery.npy"), gallery)
np.clip(np.round(gallery*127),-127,127).astype(np.int8).tofile(os.path.join(OUT,"gallery.bin"))
meta.update({"species":kept, "count":len(kept), "mu":mu.tolist(), "sd":sd.tolist()})
json.dump(meta, open(gj,"w",encoding="utf-8"), ensure_ascii=False)

pj = os.path.join(OUT,"pokedex.json")
if os.path.exists(pj):
    pdx = json.load(open(pj, encoding="utf-8"))
    pdx = {k:v for k,v in pdx.items() if is_base(k)}
    json.dump(pdx, open(pj,"w",encoding="utf-8"), ensure_ascii=False, indent=0)
mu_p, sd_p = mu, sd          # 이후 셀도 기본종만 사용
print("필터 완료 → gallery/pokedex 기본종만 저장")


In [ ]:
#@title 10. 대표 이미지 → webp 축소(결과 화면용)
import os
from PIL import Image
from tqdm.auto import tqdm
WEBP = os.path.join(OUT, "pokedex_webp"); os.makedirs(WEBP, exist_ok=True)
MAX = 512
done = skipped = 0
for sp in tqdm(kept):
    dst = os.path.join(WEBP, sp + ".webp")
    if os.path.exists(dst):            # 이미 있으면 건너뜀 → 중단돼도 재실행하면 이어서 진행
        skipped += 1; continue
    src = os.path.join(REPR_DIR, sp + ".png")
    if not os.path.exists(src): continue
    im = Image.open(src).convert("RGBA")
    im.thumbnail((MAX, MAX))
    im.save(dst, "WEBP", quality=85, method=3)   # method=6은 매우 느림 → 3(속도/용량 균형)
    done += 1
print(f"webp 저장: {done}개 (건너뜀 {skipped}) → {WEBP}")


In [ ]:
#@title 11. (선택) B2로 대표이미지 업로드 — 자격증명 붙여넣기
# B2 S3 호환 엔드포인트로 업로드. 필요한 값은 웹 프로젝트 .env.local과 동일.
DO_UPLOAD = False   #@param {type:"boolean"}
if DO_UPLOAD:
    !pip -q install boto3
    import boto3, os, glob
    from botocore.config import Config
    from tqdm.auto import tqdm
    B2_ENDPOINT = ""  #@param {type:"string"}
    B2_REGION   = ""  #@param {type:"string"}
    B2_BUCKET   = ""  #@param {type:"string"}
    B2_KEY_ID   = ""  #@param {type:"string"}
    B2_APP_KEY  = ""  #@param {type:"string"}
    s3 = boto3.client("s3", endpoint_url=B2_ENDPOINT, region_name=B2_REGION,
                      aws_access_key_id=B2_KEY_ID, aws_secret_access_key=B2_APP_KEY,
                      config=Config(signature_version="s3v4"))
    for p in tqdm(glob.glob(os.path.join(WEBP, "*.webp"))):
        key = "pokedex/" + os.path.basename(p)
        s3.upload_file(p, B2_BUCKET, key,
            ExtraArgs={"ContentType":"image/webp"})
    print("업로드 완료 → pokedex/<slug>.webp")


In [ ]:
#@title 12. 웹 자산 묶어서 다운로드
import shutil, os
from google.colab import files
BUNDLE = "/content/pokematch_web_assets"
os.makedirs(BUNDLE, exist_ok=True)
enc = "pokemon_encoder.int8.onnx" if os.path.exists(os.path.join(OUT,"pokemon_encoder.int8.onnx")) \
      else "pokemon_encoder.onnx"
for fn in [enc, "gallery.bin", "gallery.json", "pokedex.json"]:
    src = os.path.join(OUT, fn)
    if os.path.exists(src): shutil.copy(src, BUNDLE)
shutil.make_archive(BUNDLE, "zip", BUNDLE)
print("번들:", BUNDLE + ".zip")
files.download(BUNDLE + ".zip")   # 대표이미지 webp는 용량이 커서 별도(B2)로 처리


In [ ]:
#@title 13. 정합성 테스트 — 얼굴 crop + z-score 재랭킹 (앱과 동일 파이프라인)
from google.colab import files
from PIL import Image
from IPython.display import display
import torch, numpy as np, json, os, cv2
dev = next(model.parameters()).device

meta = json.load(open(os.path.join(OUT,"gallery.json"), encoding="utf-8"))
mu_p = np.array(meta["mu"]); sd_p = np.array(meta["sd"])   # 7-B에서 저장
pdx = {}; _p = os.path.join(OUT,"pokedex.json")
if os.path.exists(_p): pdx = json.load(open(_p, encoding="utf-8"))
cas = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

up = files.upload()   # 여러 장 한 번에 올려 비교 가능
for path in up:
    img = Image.open(path).convert("RGB")
    g = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    fs = cas.detectMultiScale(g, 1.1, 5, minSize=(80,80))
    if len(fs):
        x,y,w,h = sorted(fs, key=lambda b:b[2]*b[3])[-1]; cx,cy,s = x+w/2,y+h/2,max(w,h)*1.4
        img = img.crop((int(cx-s/2),int(cy-s/2),int(cx+s/2),int(cy+s/2)))
    display(img)
    with torch.no_grad():
        q = model.encode_image(preprocess(img).unsqueeze(0).to(dev)).float()
        q = torch.nn.functional.normalize(q, dim=-1).cpu().numpy()[0]
    z = (gallery @ q - mu_p) / sd_p          # ★ 인기편향 제거 z-score
    print("="*44, path)
    for i in np.argsort(-z)[:5]:
        e = pdx.get(kept[i], {})
        print(f"  z={z[i]:.2f}  #{e.get('dex')} {e.get('nameKo')} ({kept[i]})  {e.get('typesKo')}")


In [ ]:
#@title 14. (실험) 얼굴 crop(OpenCV) + 일관성/닮음 테스트
# 목표: (1) 같은 사람 여러 장 → 비슷한 결과(일관성), (2) 사람마다 그럴듯(닮음).
# ⚠️ facenet-pytorch는 쓰지 말 것 — numpy를 다운그레이드해 런타임(PIL 등)을 깨뜨린다.
#    OpenCV(기본 설치)로 얼굴 검출/crop 하면 아무것도 안 깨진다.
import numpy as np, torch, json, os, cv2
from PIL import Image
from IPython.display import display
from google.colab import files

dev = next(model.parameters()).device
GJ = os.path.join(OUT, "gallery.json")
meta = json.load(open(GJ, encoding="utf-8"))

try: gallery, kept                       # 런타임 재시작 후 없으면 디스크에서 복원
except NameError:
    gallery = np.load(os.path.join(OUT,"gallery.npy")).astype("float32"); kept = meta["species"]

if "mu" in meta and "sd" in meta:        # 7-B에서 저장됨
    mu_p = np.array(meta["mu"]); sd_p = np.array(meta["sd"])
else:                                    # 없으면 즉석 계산·저장(7-B 안 돌렸어도 동작)
    print("gallery.json에 μ_p/σ_p 없음 → LFW로 계산(최초 1회, ~200MB)")
    from sklearn.datasets import fetch_lfw_people
    lfw = fetch_lfw_people(color=True, resize=1.0, funneled=True)
    idx = np.random.default_rng(0).choice(len(lfw.images), size=min(500,len(lfw.images)), replace=False)
    F=[]
    with torch.no_grad():
        for i in idx:
            im=Image.fromarray((lfw.images[i]*255).astype("uint8")).convert("RGB")
            e=model.encode_image(preprocess(im).unsqueeze(0).to(dev)).float()
            F.append(torch.nn.functional.normalize(e,dim=-1).cpu().numpy()[0])
    A=np.stack(F)@gallery.T; mu_p=A.mean(0); sd_p=A.std(0)+1e-6
    meta["rerank"]="zscore"; meta["mu"]=mu_p.tolist(); meta["sd"]=sd_p.tolist()
    json.dump(meta, open(GJ,"w",encoding="utf-8"), ensure_ascii=False)
    print("저장 완료 (faces:", len(F), ")")

pdx = {}; _p = os.path.join(OUT,"pokedex.json")
if os.path.exists(_p): pdx = json.load(open(_p, encoding="utf-8"))

# 기본종만 노출(메가·패러독스·특수폼 제외). 9-B로 이미 걸렀으면 사실상 no-op.
PARADOX = {"great_tusk","scream_tail","brute_bonnet","flutter_mane","slither_wing",
           "sandy_shocks","roaring_moon","walking_wake","gouging_fire","raging_bolt"}
def is_base(s): return not (s.startswith(("mega_","primal_","gmax_","iron_")) or s in PARADOX)
keep_mask = np.array([is_base(s) for s in kept])

cas = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
def crop_face(img):
    g = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    fs = cas.detectMultiScale(g, 1.1, 5, minSize=(80,80))
    if len(fs) == 0: return img
    x,y,w,h = sorted(fs, key=lambda b:b[2]*b[3])[-1]
    cx,cy,s = x+w/2, y+h/2, max(w,h)*1.3
    return img.crop((int(cx-s/2), int(cy-s/2), int(cx+s/2), int(cy+s/2)))

def embed(img):
    with torch.no_grad():
        e = model.encode_image(preprocess(img).unsqueeze(0).to(dev)).float()
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()[0]

up = files.upload()   # 같은 사람 여러 장 + 다른 사람들 한꺼번에
for path in sorted(up):
    c = crop_face(Image.open(path).convert("RGB"))
    t = c.copy(); t.thumbnail((160,160)); display(t)
    z = (gallery @ embed(c) - mu_p) / sd_p
    z = np.where(keep_mask, z, -1e9)     # 메가·패러독스 배제
    print("="*40, path)
    for i in np.argsort(-z)[:5]:
        e = pdx.get(kept[i], {})
        print(f"  z={z[i]:.2f}  #{e.get('dex')} {e.get('nameKo')} ({kept[i]})  {e.get('color')}/{e.get('shape')}  {e.get('typesKo')}")


## ✅ 다음 단계

**핵심 방법 = 임베딩 최근접 + 인기편향 제거 z-score 재랭킹.** 원시 코사인은 '파라스처럼
모두와 가까운' 포켓몬이 항상 1등이 되므로, `z = (코사인 − μ_p) / σ_p` 로 재랭킹해야
사람마다 개성 있는 결과가 나온다(μ_p·σ_p는 7-B에서 gallery.json에 저장).

1. `pokematch_web_assets.zip`(encoder + gallery.bin + **μ_p·σ_p 담긴 gallery.json** + pokedex.json)
   → 웹 `/public/pokemon/`(onnx는 `/public/models/`)에 배치.
2. 11번 셀 또는 B2 UI로 `pokedex_webp/*.webp` → B2 `pokedex/`에 업로드.
3. 웹 얼굴 전처리 상수(5번 셀): **256 리사이즈·크롭, 정규화 없음([0,1])**.
4. 웹 추론: 얼굴 crop → 인코더 임베딩 → `z=(gallery·q − μ_p)/σ_p` 상위 5 → 타입색 배지·결과 저장.

미확인 도감번호/한글명은 9번 셀의 예외 매핑(`fix`, `api_name`)을 보완하면 줄일 수 있다.
